# درس ۰۴ - الگوی طراحی استفاده از ابزار

در این درس الگوی طراحی **استفاده از ابزار** برای عوامل هوش مصنوعی با استفاده از Microsoft Agent Framework (پایتون) را خواهید آموخت. موارد زیر را پوشش می‌دهیم:

- تعریف ابزارهای تابع با دکوراتور `@tool` و پارامترهای تایپ شده
- ارائه اسکیماهای ابزار تا مدل بداند هر ابزار چه کاری انجام می‌دهد
- کنترل اجرای ابزار با `approval_mode`
- بازگرداندن **خروجی ساخت‌یافته** از طریق مدل‌های Pydantic و `response_format`

سناریو یک **عامل رزرو سفر** است که می‌تواند مقصدها را جستجو کند، در دسترس بودن را بررسی نماید و اطلاعات پرواز را بازیابی کند.


## راه‌اندازی


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## تعریف ابزارها با دکوراتور @tool

دکوراتور `@tool` یک تابع ساده پایتون را به ابزاری تبدیل می‌کند که یک عامل می‌تواند آن را فراخوانی کند.  
نکات کلیدی:

- **داکیومنت استرینگ** به توضیح ابزار تبدیل می‌شود که مدل مشاهده می‌کند.  
- **آنتوتیشن‌های نوع** (شامل `Annotated` با توضیحات) قالب ابزار را تعریف می‌کنند.  
- `approval_mode` کنترل می‌کند که آیا کاربر باید قبل از اجرای هر فراخوانی، آن را تأیید کند یا خیر.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ایجاد یک عامل با چند ابزار

هر سه ابزار را به کلاینت منتقل کنید تا مدل بتواند هر کدام را که برای پاسخ به سؤال کاربر نیاز دارد فراخوانی کند.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## خروجی ساخت‌یافته با ابزارها

با تنظیم `response_format` روی یک مدل Pydantic، عامل مجبور می‌شود که یک شیء JSON با نوع‌دهی دقیق به جای متن آزاد بازگرداند. این زمانی مفید است که کدهای پایین‌دستی نیاز دارند نتیجه را به صورت برنامه‌نویسی‌شده استفاده کنند.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## الگوهای تایید ابزار

پارامتر `approval_mode` در `@tool` کنترل می‌کند که آیا فراخوانی‌های ابزار قبل از اجرا نیاز به تایید انسانی دارند یا خیر:

| حالت | رفتار |
|---|---|
| `"never_require"` | ابزار به‌صورت خودکار اجرا می‌شود — نیازی به تایید کاربر نیست. |
| `"always_require"` | هر فراخوانی باید قبل از اجرا توسط کاربر تایید شود. |

از `"always_require"` برای ابزارهایی استفاده کنید که اثرات جانبی دارند (مثلاً رزرو پرواز، شارژ کارت اعتباری) تا یک انسان در جریان باشد.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## خلاصه

در این درس یاد گرفتید چگونه:

1. **تعریف ابزارها** با استفاده از دکوراتور `@tool` با پارامترهای تایپ شده و داک‌استرینگ‌هایی که به عنوان اسکیمای ابزار عمل می‌کنند.
2. **ترکیب چندین ابزار** به طوری که عامل بتواند آنها را به ترتیب فراخوانی کند تا به پرسش‌های پیچیده پاسخ دهد.
3. **بازگرداندن خروجی ساختارمند** با ارسال یک مدل Pydantic به عنوان `response_format`.
4. **کنترل تایید ابزار** با `approval_mode` برای نگه داشتن انسان در حلقه برای عملیات حساس.

این الگوها پایه و اساس ساخت عوامل قابل اعتماد و آماده برای تولید را تشکیل می‌دهند که می‌توانند با سیستم‌های خارجی به طور ایمن تعامل داشته باشند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**توضیح مهم**:  
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در پی دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی اشتباهات یا نادرستی‌هایی باشند. سند اصلی به زبان بومی خود باید به‌عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حساس، توصیه می‌شود از ترجمه حرفه‌ای انسانی استفاده نمایید. ما مسئول هیچ گونه سوءتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
